# Attribute Transformations: Shaping Data for Machine Learning

## 1. Clear Overview

Attribute transformations involve applying mathematical or logical functions to existing features to reshape their distribution. By converting raw data into more symmetric, bell-shaped forms, these techniques help models (particularly linear algorithms) identify patterns more effectively, reduce the impact of extreme outliers, and improve overall predictive stability.

In [ ]:
# Always start with imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Scikit-Learn transformers and models
from sklearn.preprocessing import PowerTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
# Set display options for pandas
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)

# Set plotting aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Set random seed for reproducibility
np.random.seed(42)

print("Step 1: All necessary libraries successfully imported and configured!")

## 2. Structured Table of Contents

- **Synthetic Data Creation**: Generating realistically skewed features
- **Why Transformations are Necessary**: Understanding model assumptions
- **Skewness Explained**: Positive vs. Negative asymmetry
- **Core Concept 1**: Log Transformation
- **Core Concept 2**: Power Transformation (Yeo-Johnson)
- **Core Concept 3**: Handling Negative Values & Zeros
- **Impact on Machine Learning**: Proving transformations work
- **Visualization Gallery**: Comparative plotting matrix
- **Practice Exercises**: Test your understanding
- **Application Summary**: Best practices

## 3. Synthetic Data Creation

To effectively demonstrate transformation techniques, we will construct a synthetic dataset. We will intentionally create features with distinct distributional problems:

1. free_sulfur_dioxide: Strongly right-skewed (exponentially distributed) to simulate trace chemical quantities.
2. temperature_change: Normally distributed but shifted to contain negative values (to demonstrate advanced power transformations).
3. quality_score: Strongly left-skewed (beta distributed) to simulate customer ratings where most users give high scores.

In [ ]:
# Generate synthetic data
n_samples = 2000

# 1. Right-skewed data (always positive)
free_sulfur_dioxide = np.random.exponential(scale=15.0, size=n_samples)

# 2. Data with negatives and zeros
temperature_change = np.random.normal(loc=2.0, scale=10.0, size=n_samples) - 8.0

# 3. Left-skewed data
quality_score = np.random.beta(a=8, b=2, size=n_samples) * 100

# Target Variable: We define the hidden relationship linearly against the LOG of sulfur dioxide
# This guarantees that transforming the feature will improve model performance later
target = (12.5 * np.log1p(free_sulfur_dioxide)) + (1.2 * temperature_change) + np.random.normal(0, 4.0, n_samples)

# Assemble DataFrame
df = pd.DataFrame({
    'free_sulfur_dioxide': free_sulfur_dioxide,
    'temperature_change': temperature_change,
    'quality_score': quality_score,
    'target': target
})

print("Synthetic dataset generated successfully!")

In [ ]:
print("--- Dataset Information ---")
print(df.info())

print("\n--- First 5 Rows ---")
print(df.head())

print("\n--- Summary Statistics ---")
print(df.describe().round(2))

## 4. Why Transformations are Necessary & Skewness Explained

Raw data rarely fits the ideal statistical assumptions required by many algorithms. Transformations are used to:

- **Reduce Skewness:** Converting skewed distributions into more symmetric, normal-like distributions.
- **Handle Outliers:** Compressing the range of extreme values so they do not disproportionately bias the model.
- **Highlight Relationships:** Reshaping variables to make nonlinear relationships more apparent to linear models.

### Skewness Explained
- **Positive (Right) Skew:** The tail is longer on the right side. Most data points are concentrated on the left (lower values). The mean is pulled to the right.
- **Negative (Left) Skew:** The tail is longer on the left side. Most data points are on the right (higher values). The mean is pulled to the left.

In [ ]:
# Calculate numerical skewness
skewness_vals = df.skew().round(3)
print("--- Skewness Values ---")
print(skewness_vals)
print("\nRule of Thumb: Skewness between -0.5 and 0.5 is fairly symmetrical.")
print("Skewness > 1 or < -1 indicates a highly skewed distribution.")

In [ ]:
# Visualize the raw skewed distribution
plt.figure(figsize=(10, 5))
sns.histplot(df['free_sulfur_dioxide'], bins=50, kde=True, color='skyblue')

plt.title('Raw Data: Right-Skewed Free Sulfur Dioxide', fontsize=14)
plt.xlabel('Free Sulfur Dioxide Concentration', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

plt.axvline(df['free_sulfur_dioxide'].mean(), color='red', linestyle='--', label='Mean')
plt.axvline(df['free_sulfur_dioxide'].median(), color='green', linestyle='-', label='Median')

plt.legend()
plt.tight_layout()
plt.show()

print("Observation: The mean is significantly pulled to the right of the median by extreme outliers.")

## 5. Core Concept 1: Log Transformation

The **Log Transformation** uses the natural logarithm log(1+x) to compress large values more significantly than small values.

- **Mechanism:** Applying log effectively pulls in extreme right-tail values while stretching out smaller values.
- **Pros:** Simple, highly interpretable, standard practice for exponential growth or decay data.
- **Caveat:** Typically requires all values to be non-negative. We use numpy's `log1p` (which calculates log(1+x)) to safely handle exact zero-value inputs without generating negative infinity errors.

In [ ]:
# Apply Log Transformation using np.log1p()
df['fsd_log'] = np.log1p(df['free_sulfur_dioxide'])

print("Log Transformation Applied!")
print(f"Original Skewness: {df['free_sulfur_dioxide'].skew():.3f}")
print(f"Transformed Skewness: {df['fsd_log'].skew():.3f}")

### Visualizing the Log Transformation
Let's observe how the exponential curve collapses into a beautiful, normal-looking bell curve.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Raw Data
sns.histplot(df['free_sulfur_dioxide'], bins=50, kde=True, ax=axes[0], color='lightgray')
axes[0].set_title('Before: Raw Right Skew', fontsize=14)
axes[0].set_xlabel('Free Sulfur Dioxide')

# Plot 2: Log Transformed
sns.histplot(df['fsd_log'], bins=50, kde=True, ax=axes[1], color='blue')
axes[1].set_title('After: Log1p Transformation', fontsize=14)
axes[1].set_xlabel('log(1 + Free Sulfur Dioxide)')

plt.tight_layout()
plt.show()

print("Success: The data now strongly resembles a normal distribution, making it perfect for Linear Regression.")

## 6. Core Concept 2: Power Transformation (Yeo-Johnson)

The **Power Transformation** (specifically the Yeo-Johnson method) is an automated approach that searches for the optimal lambda parameter to force a distribution toward normality.

- **Mechanism:** It applies different power functions based on the data's characteristics.
- **Pros:** Much more flexible than simple log transforms. It adapts dynamically to the exact shape of the skewness.

In [ ]:
# Apply Yeo-Johnson Power Transformation using Scikit-Learn
pt_yj = PowerTransformer(method='yeo-johnson')

# Note: Scikit-Learn transformers expect a 2D array, so we pass df[['column_name']]
df['fsd_yj'] = pt_yj.fit_transform(df[['free_sulfur_dioxide']])

print("Yeo-Johnson Transformation Applied!")
print(f"Optimal Lambda discovered by the algorithm: {pt_yj.lambdas_[0]:.3f}")
print(f"Yeo-Johnson Skewness: {df['fsd_yj'].skew():.3f}")

### Visualizing the Yeo-Johnson Transformation

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df['fsd_yj'], bins=50, kde=True, color='purple')

plt.title('Yeo-Johnson Transformation Result', fontsize=14)
plt.xlabel('Standardized Yeo-Johnson Value')
plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

print("Note: PowerTransformer automatically standardizes the output (mean=0, variance=1) as part of the process.")

## 7. Core Concept 3: Handling Negative Values & Zeros

Why do we need Yeo-Johnson if log transformations exist? 

Logarithms are mathematically undefined for negative numbers. If your feature contains sub-zero values (like our `temperature_change` column), a simple log transformation will crash or produce NaNs.

Yeo-Johnson is explicitly designed to handle negative values safely.

In [ ]:
# Demonstrate the robust nature of Yeo-Johnson on negative data
print(f"Minimum value in temperature_change: {df['temperature_change'].min():.2f}")

# This would throw a warning/error if we tried np.log()
pt_temp = PowerTransformer(method='yeo-johnson')
df['temp_yj'] = pt_temp.fit_transform(df[['temperature_change']])

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df['temperature_change'], bins=50, kde=True, ax=axes[0], color='orange')
axes[0].set_title('Raw Temperature (Includes Negatives)', fontsize=14)

sns.histplot(df['temp_yj'], bins=50, kde=True, ax=axes[1], color='teal')
axes[1].set_title('Yeo-Johnson Transformed', fontsize=14)

plt.tight_layout()
plt.show()

print("Success: Yeo-Johnson securely transformed data spanning both negative and positive domains.")

## 8. Impact on Machine Learning Models

Transformations are not inherently 'good' or 'bad'; they must be validated through model performance metrics. If the transformation does not improve the model's accuracy, the original feature may be preferable.

Let's train two simple Ridge Regression models to predict our target variable:
1. Using strictly raw, un-transformed features.
2. Using the logically transformed feature.

In [ ]:
# Define features sets
X_raw = df[['free_sulfur_dioxide', 'temperature_change']]
X_transformed = df[['fsd_log', 'temperature_change']]
y = df['target']

# Split into Train and Test
X_raw_tr, X_raw_te, y_tr, y_te = train_test_split(X_raw, y, test_size=0.2, random_state=42)
X_trans_tr, X_trans_te, _, _ = train_test_split(X_transformed, y, test_size=0.2, random_state=42)

# Train Model 1: Raw Data
model_raw = Ridge(alpha=1.0)
model_raw.fit(X_raw_tr, y_tr)
preds_raw = model_raw.predict(X_raw_te)
r2_raw = r2_score(y_te, preds_raw)

# Train Model 2: Transformed Data
model_trans = Ridge(alpha=1.0)
model_trans.fit(X_trans_tr, y_tr)
preds_trans = model_trans.predict(X_trans_te)
r2_trans = r2_score(y_te, preds_trans)

print("--- Model Evaluation (R-Squared Score) ---")
print(f"Raw Features Model:         {r2_raw:.4f}")
print(f"Transformed Features Model: {r2_trans:.4f}")

print("\nConclusion: The transformed feature unlocked the hidden linear relationship, vastly improving model accuracy!")

## 9. Visualization Gallery

Let's create a comparative matrix to see all our transformations side by side.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Top-Left: Raw
sns.histplot(df['free_sulfur_dioxide'], kde=True, ax=axes[0, 0], color='gray')
axes[0, 0].set_title('Raw Skewed Data', fontsize=14)

# Top-Right: Log1p
sns.histplot(df['fsd_log'], kde=True, ax=axes[0, 1], color='blue')
axes[0, 1].set_title('Log(1+x) Transformation', fontsize=14)

# Bottom-Left: Square Root (A milder alternative to Log)
sns.histplot(np.sqrt(df['free_sulfur_dioxide']), kde=True, ax=axes[1, 0], color='green')
axes[1, 0].set_title('Square Root Transformation', fontsize=14)

# Bottom-Right: Yeo-Johnson
sns.histplot(df['fsd_yj'], kde=True, ax=axes[1, 1], color='purple')
axes[1, 1].set_title('Yeo-Johnson Transformation', fontsize=14)

plt.suptitle('Comprehensive Transformation Gallery', fontsize=18, y=1.02)
plt.tight_layout()
plt.show()

## 10. Practice Exercises

Now it is your turn to apply these tools to new scenarios.

### Exercise 1: Handling Left-Skewed Data

**Task:** 
Look at the `quality_score` column. It is strongly left-skewed. Log transformations deal with right-skewed data. 
Can you apply a Yeo-Johnson transformation to fix this left-skewness? Calculate the skewness before and after.

In [ ]:
# --- EXERCISE 1 SOLUTION ---

print(f"Raw Left-Skewness: {df['quality_score'].skew():.3f}")

# Initialize and apply Yeo-Johnson
pt_left = PowerTransformer(method='yeo-johnson')
df['quality_yj'] = pt_left.fit_transform(df[['quality_score']])

print(f"Transformed Skewness: {df['quality_yj'].skew():.3f}")

print("\nNotice how Yeo-Johnson automatically detects the left-skew and adjusts its power parameter to pull the left tail inward!")

### Exercise 2: Visualizing the Exercise Result

**Task:** 
Create a side-by-side histogram comparing the raw `quality_score` with your new `quality_yj` feature.

In [ ]:
# --- EXERCISE 2 SOLUTION ---

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df['quality_score'], bins=30, kde=True, ax=axes[0], color='darkred')
axes[0].set_title('Before: Strong Left Skew', fontsize=14)
axes[0].set_xlabel('Quality Score')

sns.histplot(df['quality_yj'], bins=30, kde=True, ax=axes[1], color='darkgreen')
axes[1].set_title('After: Yeo-Johnson Corrected', fontsize=14)
axes[1].set_xlabel('Transformed Quality')

plt.tight_layout()
plt.show()

## 11. Application Summary & Key Takeaways

- **Decision Framework:** No single transformation fits every dataset. Practitioners must visualize the distribution pre- and post-transformation to ensure intended behavior.
- **Log for Magnitude:** Use `np.log1p()` for simple right-skewed positive data, especially when values span several orders of magnitude (like salary, website traffic, or chemical concentrations).
- **Yeo-Johnson for Complexity:** Use Scikit-Learn's `PowerTransformer` when dealing with negative values, complex shapes, or when you need automated pipeline integration.
- **Validation is Mandatory:** Always validate your transformations against a model. If transforming the feature degrades validation metrics, revert to the raw feature.

In [ ]:
print("-----------------------------------------------------------")
print("Notebook Execution Complete.")
print("You have successfully mastered Attribute Transformations!")
print("-----------------------------------------------------------")